<a href="https://colab.research.google.com/github/naseealavutheen1805/Python-project/blob/main/E_Commerce_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# E-COMMERCE ANALYSIS - GOOGLE COLAB SCRIPT

from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Upload your CSV file
print("📁 Please upload your CSV file...")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"✅ File uploaded: {filename}")

📁 Please upload your CSV file...


Saving compressed_data.csv.gz to compressed_data.csv.gz
✅ File uploaded: compressed_data.csv.gz


In [ ]:
# ============================================================================

# Load data
df = pd.read_csv(filename, low_memory=False)

print("="*80)
print("📊 DATASET OVERVIEW")
print("="*80)
print(f"\nShape: {df.shape}")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

# Display first rows
print("\n📋 First 5 rows:")
display(df.head())

# Column info
print("\n📝 Column Names:")
print(df.columns.tolist())

# ============================================================================
print("="*80)
print("🔍 DATA QUALITY ASSESSMENT")
print("="*80)

# Missing values
missing = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_%': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

missing = missing[missing['Missing_Count'] > 0]
print("\nMissing Values:")
display(missing)

# Data types
print("\n📊 Data Types:")
print(df.dtypes)


# ============================================================================
print("="*80)
print("🧹 DATA CLEANING - STEP 1")
print("="*80)

# Remove unnamed column
if 'Unnamed: 22' in df.columns:
    df = df.drop('Unnamed: 22', axis=1)
    print("✅ Removed 'Unnamed: 22' column")

# Rename columns
df = df.rename(columns={
    'Sales Channel ': 'Sales_Channel',
    'Qty': 'Quantity',
    'Fulfilment': 'Fulfillment',
    'ship-service-level': 'Service_Level',
    'ship-city': 'City',
    'ship-state': 'State',
    'ship-postal-code': 'Postal_Code',
    'ship-country': 'Country',
    'promotion-ids': 'Promotion_IDs',
    'fulfilled-by': 'Fulfilled_By',
    'Courier Status': 'Courier_Status'
})

print("✅ Columns renamed")
print(f"New shape: {df.shape}")

# ============================================================================
print("="*80)
print("🧹 DATA CLEANING - STEP 2 (DATE)")
print("="*80)

# Convert date
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')

print(f"✅ Date column converted")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Total days: {(df['Date'].max() - df['Date'].min()).days}")

# ============================================================================
print("="*80)
print("🧹 DATA CLEANING - STEP 3 (MISSING VALUES)")
print("="*80)

# Calculate median for Amount
amount_median = df['Amount'].median()
print(f"Amount median: ₹{amount_median:.2f}")

# Replace nulls in Amount with median (as per problem statement)
nulls_before = df['Amount'].isnull().sum()
df['Amount'] = df['Amount'].fillna(amount_median)
nulls_after = df['Amount'].isnull().sum()

print(f"✅ Replaced {nulls_before:,} null values in Amount with median")
print(f"Null values remaining: {nulls_after}")

# Fill other missing values
df['currency'] = df['currency'].fillna('INR')
df['Courier_Status'] = df['Courier_Status'].fillna('Not Available')
df['City'] = df['City'].fillna('Unknown')
df['State'] = df['State'].fillna('Unknown')

print("✅ All critical missing values handled")

# ============================================================================
print("="*80)
print("🧹 DATA CLEANING - STEP 4 (CALCULATED COLUMNS)")
print("="*80)

# Customer Type
df['Customer_Type'] = df['B2B'].apply(lambda x: 'B2B' if x == True else 'B2C')

# Status Group
def categorize_status(status):
    if pd.isna(status):
        return 'Other'
    elif 'Cancelled' in str(status):
        return 'Cancelled'
    elif 'Delivered' in str(status):
        return 'Delivered'
    elif 'Return' in str(status) or 'Reject' in str(status):
        return 'Returned'
    elif status == 'Shipped':
        return 'Shipped'
    elif 'Pending' in str(status):
        return 'Pending'
    else:
        return 'Other'

df['Status_Group'] = df['Status'].apply(categorize_status)

# Date components
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%B')
df['Month_Year'] = df['Date'].dt.strftime('%b %Y')
df['Week'] = df['Date'].dt.isocalendar().week

# Has Promotion
df['Has_Promotion'] = df['Promotion_IDs'].notna().map({True: 'Yes', False: 'No'})

# Revenue
df['Revenue'] = df['Amount']

print("✅ Created calculated columns:")
print("   - Customer_Type")
print("   - Status_Group")
print("   - Date components")
print("   - Has_Promotion")
print("   - Revenue")

print(f"\n✅ DATA CLEANING COMPLETE!")
print(f"Final shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")

# ============================================================================
print("\n" + "="*80)
print("💰 FINANCIAL PERFORMANCE ANALYSIS")
print("="*80)

# Key metrics
total_revenue = df['Revenue'].sum()
total_orders = len(df)
avg_order_value = df['Revenue'].mean()
total_quantity = df['Quantity'].sum()

print(f"\n💰 Total Revenue: ₹{total_revenue:,.2f}")
print(f"📦 Total Orders: {total_orders:,}")
print(f"💵 Average Order Value: ₹{avg_order_value:.2f}")
print(f"📊 Total Units Sold: {total_quantity:,}")

# Monthly revenue
monthly_revenue = df.groupby('Month_Year').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index()
monthly_revenue.columns = ['Month_Year', 'Revenue', 'Orders']

print("\n📅 Monthly Performance:")
display(monthly_revenue)

# Revenue trend over time
daily_revenue = df.groupby('Date')['Revenue'].sum().reset_index()

fig = px.line(daily_revenue, x='Date', y='Revenue',
              title='Daily Revenue Trend',
              labels={'Revenue': 'Revenue (₹)'})
fig.update_layout(height=500)
fig.show()

# Monthly revenue bar chart
fig = px.bar(monthly_revenue, x='Month_Year', y='Revenue',
             title='Monthly Revenue Distribution',
             labels={'Revenue': 'Revenue (₹)'},
             color='Revenue',
             color_continuous_scale='Blues')
fig.update_layout(height=500)
fig.show()

# ============================================================================
print("\n" + "="*80)
print("📊 CATEGORY ANALYSIS")
print("="*80)

# Category performance
category_revenue = df.groupby('Category').agg({
    'Revenue': 'sum',
    'Order ID': 'count',
    'Quantity': 'sum'
}).reset_index().sort_values('Revenue', ascending=False)

category_revenue.columns = ['Category', 'Revenue', 'Orders', 'Units']
category_revenue['AOV'] = category_revenue['Revenue'] / category_revenue['Orders']
category_revenue['Revenue_%'] = (category_revenue['Revenue'] / total_revenue * 100).round(2)

print("\n💰 Revenue by Category:")
display(category_revenue)

# Visualize
fig = px.pie(category_revenue, values='Revenue', names='Category',
             title='Revenue Distribution by Category',
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=600)
fig.show()

# Top 20 products
top_products = df.groupby(['SKU', 'Category']).agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index().sort_values('Revenue', ascending=False).head(20)

top_products.columns = ['SKU', 'Category', 'Revenue', 'Orders']

print("\n🏆 Top 20 Products by Revenue:")
display(top_products)

# Visualize
fig = px.bar(top_products, x='Revenue', y='SKU', orientation='h',
             title='Top 20 Products by Revenue',
             color='Category',
             height=700)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# ============================================================================
print("\n" + "="*80)
print("👥 CUSTOMER INSIGHTS ANALYSIS")
print("="*80)

# B2B vs B2C
b2b_analysis = df.groupby('Customer_Type').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index()

b2b_analysis.columns = ['Customer_Type', 'Revenue', 'Orders']
b2b_analysis['AOV'] = b2b_analysis['Revenue'] / b2b_analysis['Orders']
b2b_analysis['Revenue_%'] = (b2b_analysis['Revenue'] / total_revenue * 100).round(2)

print("\n💼 B2B vs B2C Performance:")
display(b2b_analysis)

# Visualize
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Revenue Distribution', 'Order Distribution'),
                    specs=[[{'type':'pie'}, {'type':'pie'}]])

fig.add_trace(
    go.Pie(labels=b2b_analysis['Customer_Type'], values=b2b_analysis['Revenue'],
           name="Revenue"),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=b2b_analysis['Customer_Type'], values=b2b_analysis['Orders'],
           name="Orders"),
    row=1, col=2
)

fig.update_layout(title_text="B2B vs B2C Comparison", height=500)
fig.show()

# ============================================================================
print("\n" + "="*80)
print("🗺️ GEOGRAPHIC ANALYSIS")
print("="*80)

# Top states
state_performance = df.groupby('State').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index().sort_values('Revenue', ascending=False).head(15)

state_performance.columns = ['State', 'Revenue', 'Orders']
state_performance['AOV'] = state_performance['Revenue'] / state_performance['Orders']

print("\n🏙️ Top 15 States by Revenue:")
display(state_performance)

# Visualize
fig = px.bar(state_performance, x='State', y='Revenue',
             title='Top 15 States by Revenue',
             color='Revenue',
             color_continuous_scale='Viridis')
fig.update_layout(height=600, xaxis_tickangle=-45)
fig.show()

# Top cities
city_performance = df.groupby('City').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index().sort_values('Revenue', ascending=False).head(15)

city_performance.columns = ['City', 'Revenue', 'Orders']

print("\n🏙️ Top 15 Cities by Revenue:")
display(city_performance)

# Visualize
fig = px.bar(city_performance, y='City', x='Revenue', orientation='h',
             title='Top 15 Cities by Revenue',
             color='Revenue',
             color_continuous_scale='Plasma')
fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.show()

# ============================================================================
print("\n" + "="*80)
print("🚚 LOGISTICS & FULFILLMENT ANALYSIS")
print("="*80)

# Fulfillment type
fulfillment_stats = df.groupby('Fulfillment').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index()

fulfillment_stats.columns = ['Fulfillment', 'Revenue', 'Orders']
fulfillment_stats['AOV'] = fulfillment_stats['Revenue'] / fulfillment_stats['Orders']
fulfillment_stats['Orders_%'] = (fulfillment_stats['Orders'] / total_orders * 100).round(2)

print("\n📦 Fulfillment Performance:")
display(fulfillment_stats)

# Service level
service_stats = df.groupby('Service_Level').agg({
    'Revenue': 'sum',
    'Order ID': 'count'
}).reset_index()

service_stats.columns = ['Service_Level', 'Revenue', 'Orders']
service_stats['Orders_%'] = (service_stats['Orders'] / total_orders * 100).round(2)

print("\n🚀 Service Level Distribution:")
display(service_stats)

# Visualize
fig = px.pie(service_stats, values='Orders', names='Service_Level',
             title='Service Level Distribution',
             hole=0.3)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

# ============================================================================
print("\n" + "="*80)
print("⚠️ QUALITY & RETURNS ANALYSIS")
print("="*80)

# Status distribution
status_distribution = df.groupby('Status_Group').agg({
    'Order ID': 'count',
    'Revenue': 'sum'
}).reset_index().sort_values('Order ID', ascending=False)

status_distribution.columns = ['Status_Group', 'Orders', 'Revenue']
status_distribution['Orders_%'] = (status_distribution['Orders'] / total_orders * 100).round(2)

print("\n📊 Order Status Distribution:")
display(status_distribution)

# Calculate key rates
cancelled_orders = status_distribution[status_distribution['Status_Group'] == 'Cancelled']['Orders'].values
cancelled_orders = cancelled_orders[0] if len(cancelled_orders) > 0 else 0

cancellation_rate = (cancelled_orders / total_orders * 100)

print(f"\n⚠️ Cancellation Rate: {cancellation_rate:.2f}%")

# Visualize
fig = px.pie(status_distribution, values='Orders', names='Status_Group',
             title='Order Status Distribution',
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=600)
fig.show()

# ============================================================================
print("\n" + "="*80)
print("📊 CANCELLATION ANALYSIS BY CATEGORY")
print("="*80)

# Category cancellation rate
category_totals = df.groupby('Category').size().reset_index(name='Total_Orders')
category_cancelled = df[df['Status_Group'] == 'Cancelled'].groupby('Category').size().reset_index(name='Cancelled_Orders')

category_cancellation = category_totals.merge(category_cancelled, on='Category', how='left').fillna(0)
category_cancellation['Cancellation_Rate_%'] = (
    category_cancellation['Cancelled_Orders'] / category_cancellation['Total_Orders'] * 100
).round(2)
category_cancellation = category_cancellation.sort_values('Cancellation_Rate_%', ascending=False)

print("\n⚠️ Cancellation Rate by Category:")
display(category_cancellation)

# Visualize
fig = px.bar(category_cancellation, x='Category', y='Cancellation_Rate_%',
             title='Cancellation Rate by Category',
             color='Cancellation_Rate_%',
             color_continuous_scale='Reds')
fig.add_hline(y=cancellation_rate, line_dash="dash", line_color="red",
              annotation_text=f"Overall Avg: {cancellation_rate:.2f}%")
fig.update_layout(height=600, xaxis_tickangle=-45)
fig.show()

# ============================================================================
print("\n" + "="*80)
print("🔍 PROBLEM PRODUCTS ANALYSIS")
print("="*80)

# Products with high cancellation
product_totals = df.groupby(['SKU', 'Category']).size().reset_index(name='Total_Orders')
product_cancelled = df[df['Status_Group'] == 'Cancelled'].groupby(['SKU', 'Category']).size().reset_index(name='Cancelled_Orders')

product_quality = product_totals.merge(product_cancelled, on=['SKU', 'Category'], how='left').fillna(0)
product_quality['Cancellation_Rate_%'] = (
    product_quality['Cancelled_Orders'] / product_quality['Total_Orders'] * 100
).round(2)

# Problem products: >20% cancellation and at least 5 orders
problem_products = product_quality[
    (product_quality['Cancellation_Rate_%'] > 20) &
    (product_quality['Total_Orders'] >= 5)
].sort_values('Cancellation_Rate_%', ascending=False)

print(f"\n⚠️ Total Problem Products (>20% cancellation): {len(problem_products)}")
print("\nTop 20 Problem Products:")
display(problem_products.head(20))

if len(problem_products) > 0:
    # Visualize top 15
    fig = px.bar(problem_products.head(15), y='SKU', x='Cancellation_Rate_%', orientation='h',
                 title='Top 15 Problem Products (High Cancellation Rate)',
                 color='Category',
                 height=700)
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()

# ============================================================================
print("\n" + "="*80)
print("📊 COMPREHENSIVE BUSINESS SUMMARY")
print("="*80)

print("\n💰 FINANCIAL METRICS:")
print(f"   Total Revenue: ₹{total_revenue:,.2f}")
print(f"   Average Order Value: ₹{avg_order_value:.2f}")
print(f"   Total Orders: {total_orders:,}")
print(f"   Total Units Sold: {total_quantity:,}")

print("\n👥 CUSTOMER METRICS:")
print(f"   Total Customers: {df['Postal_Code'].nunique():,}")
print(f"   B2B Revenue %: {b2b_analysis[b2b_analysis['Customer_Type']=='B2B']['Revenue_%'].values[0]:.2f}%")
print(f"   States Covered: {df['State'].nunique()}")
print(f"   Cities Covered: {df['City'].nunique():,}")

print("\n🚚 LOGISTICS METRICS:")
amazon_pct = fulfillment_stats[fulfillment_stats['Fulfillment']=='Amazon']['Orders_%'].values
amazon_pct = amazon_pct[0] if len(amazon_pct) > 0 else 0
print(f"   Amazon Fulfillment: {amazon_pct:.2f}%")

expedited_pct = service_stats[service_stats['Service_Level']=='Expedited']['Orders_%'].values
expedited_pct = expedited_pct[0] if len(expedited_pct) > 0 else 0
print(f"   Expedited Shipping: {expedited_pct:.2f}%")

print("\n📦 PRODUCT METRICS:")
print(f"   Total Products (SKUs): {df['SKU'].nunique():,}")
print(f"   Product Categories: {df['Category'].nunique()}")

print("\n⚠️ QUALITY METRICS:")
print(f"   Cancellation Rate: {cancellation_rate:.2f}%")
print(f"   Cancelled Orders: {cancelled_orders:,}")
cancelled_revenue = df[df['Status_Group'] == 'Cancelled']['Revenue'].sum()
print(f"   Revenue Loss from Cancellations: ₹{cancelled_revenue:,.2f}")

print("\n" + "="*80)

# ============================================================================
print("\n" + "="*80)
print("💡 STRATEGIC RECOMMENDATIONS")
print("="*80)

print("""
🎯 IMMEDIATE ACTIONS (0-3 months):

1. Quality Control:
   • Investigate products with >20% cancellation rate
   • Implement quality checks for high-return categories
   • Improve product descriptions

2. Logistics Optimization:
   • Expand Amazon fulfillment for better performance
   • Address merchant fulfillment issues
   • Improve packaging to reduce damage

3. Customer Experience:
   • Enhance size guides
   • Faster resolution of queries
   • Proactive communication

📈 GROWTH OPPORTUNITIES (3-6 months):

1. B2B Segment Expansion:
   • Targeted campaigns (B2B has higher AOV)
   • Bulk order discounts
   • Dedicated account management

2. Geographic Expansion:
   • Focus on top 10-15 cities
   • Improve tier-2 city logistics
   • State-specific campaigns

3. Category Optimization:
   • Increase inventory of top categories
   • Phase out underperforming products
   • Introduce complementary products

🚀 STRATEGIC INITIATIVES (6-12 months):

1. Technology & Automation:
   • Real-time inventory tracking
   • Automated quality alerts
   • Predictive cancellation prevention

2. Marketing & Promotions:
   • Data-driven campaigns
   • Seasonal offers
   • Loyalty programs

💡 KEY METRICS TO MONITOR:

• Reduce cancellation rate from 15.73% to <10%
• Increase delivery rate to >90%
• Grow B2B segment revenue by 25%
• Improve Amazon fulfillment % to >70%
• Reduce revenue loss by 30%
""")

# ============================================================================
from google.colab import files

# Save cleaned dataset
df.to_csv('ecommerce_cleaned_data.csv', index=False)
print("✅ Cleaned dataset saved: ecommerce_cleaned_data.csv")

# Save analysis tables
category_revenue.to_csv('category_performance.csv', index=False)
state_performance.to_csv('state_performance.csv', index=False)
if len(problem_products) > 0:
    problem_products.to_csv('problem_products.csv', index=False)

print("\n✅ ANALYSIS COMPLETE!")
print("\n📊 Files generated:")
print("   1. ecommerce_cleaned_data.csv")
print("   2. category_performance.csv")
print("   3. state_performance.csv")
print("   4. problem_products.csv")

# Download files
print("\n⬇️ Downloading files...")
files.download('ecommerce_cleaned_data.csv')
files.download('category_performance.csv')
files.download('state_performance.csv')
if len(problem_products) > 0:
    files.download('problem_products.csv')

print("\n🎉 All done! Analysis completed successfully!")

📊 DATASET OVERVIEW

Shape: (128949, 23)
Total Records: 128,949
Total Columns: 23

📋 First 5 rows:


,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN



📝 Column Names:
['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B', 'fulfilled-by', 'Unnamed: 22']
🔍 DATA QUALITY ASSESSMENT

Missing Values:


,Column,Missing_Count,Missing_%
fulfilled-by,fulfilled-by,89679,69.55
promotion-ids,promotion-ids,49142,38.11
Unnamed: 22,Unnamed: 22,49041,38.03
currency,currency,7794,6.04
Amount,Amount,7794,6.04
Courier Status,Courier Status,6871,5.33
ship-city,ship-city,33,0.03
ship-state,ship-state,33,0.03
ship-postal-code,ship-postal-code,33,0.03
ship-country,ship-country,33,0.03



📊 Data Types:
Order ID               object
Date                   object
Status                 object
Fulfilment             object
Sales Channel          object
ship-service-level     object
Style                  object
SKU                    object
Category               object
Size                   object
ASIN                   object
Courier Status         object
Qty                     int64
currency               object
Amount                float64
ship-city              object
ship-state             object
ship-postal-code      float64
ship-country           object
promotion-ids          object
B2B                      bool
fulfilled-by           object
Unnamed: 22            object
dtype: object
🧹 DATA CLEANING - STEP 1
✅ Removed 'Unnamed: 22' column
✅ Columns renamed
New shape: (128949, 22)
🧹 DATA CLEANING - STEP 2 (DATE)
✅ Date column converted
Date range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00
Total days: 90
🧹 DATA CLEANING - STEP 3 (MISSING VALUES)
Amount median: 

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 All done! Analysis completed successfully!
